In [1]:
## Keeping this cell same 
import boto3

#Setting up the client object instance
multiturn_client = boto3.client("bedrock-runtime", region_name = "us-west-2")

# Setting up the model ID -> Important to know the difference between model id and inference id
# model id can be available in a region but not in other, hence leverage inference profile
# Here to start with we are using model id
model_id = "us.anthropic.claude-sonnet-4-20250514-v1:0"


## Since I will append messages from conversations I will create a variable of type list to store the message
messages = []

# Both function combined are making context for the model

# function take 'text' as input and format it in required format and add it to message list - this is for user input
def add_user_message(messages, text):
    user_message = {
        "role" : "user",
        "content" : [
            { "text" : text }
        ]
    }
    messages.append(user_message)

# function take 'text' as input and format it in required format and add it to message list - this is for model output
def add_assistant_message(messages, text):
    user_message = {
        "role" : "assistant",
        "content" : [
            { "text" : text }
        ]
    }
    messages.append(user_message)


## Playing with converse_stream api call now 

In [5]:
# Adding system Prompt - adding to request

# adding on more parameter - Temprature - default as 1.0
def chat(messages, system = None, temperature = 1.0):

    # Defining a variable  of dic type for parameters
    params = {
        "modelId": model_id, 
        "messages": messages,
        "inferenceConfig": {
            "temperature" : temperature
        }
    } # Defining a variable dic for parameters

    if system:  # if system prompt is passed to this function it will be add to params
        params["system"] = [{"text" : system}]

    response = multiturn_client.converse_stream(**params) # pointers to param variable # using converse_stream
    
    return response # just returning response here

In [6]:
add_user_message(messages, "Explain what was Claude?")
print(chat(messages))

{'ResponseMetadata': {'RequestId': '25a013d4-39c9-488f-bc12-98739420fe14', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 25 Aug 2025 14:40:45 GMT', 'content-type': 'application/vnd.amazon.eventstream', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'x-amzn-requestid': '25a013d4-39c9-488f-bc12-98739420fe14'}, 'RetryAttempts': 0}, 'stream': <botocore.eventstream.EventStream object at 0x7ff8ac0cbfd0>}


In [11]:
add_user_message(messages, "Explain who was Claude, the man?")
response = chat(messages)
for event in response["stream"]:
    print(event)

{'messageStart': {'role': 'assistant'}}
{'contentBlockDelta': {'delta': {'text': 'I notice'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' you\'re asking about "'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': 'Claude" repeatedly'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' with'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' slight'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' variations.'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' Since'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': " you're"}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' asking about "'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': 'Claude,'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' the man," you'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' might'}, 'cont

In [ ]:
### Following is an important code block as it gives you feel of low latency, and 

In [16]:
add_user_message(messages, "Explain who was Claude, the man?")
response = chat(messages)
for event in response["stream"]:
    if 'contentBlockDelta' in event:
        chunk = event['contentBlockDelta']['delta']['text']
        print(chunk, end="")

I notice you've repeated the question several times with slight variations. I'm Claude, an AI assistant created by Anthropic. 

If you're asking about me specifically, I'm an AI designed to be helpful, harmless, and honest in conversations. I can assist with a wide variety of tasks like answering questions, writing, analysis, math, coding, and creative projects.

If you're asking about someone named Claude other than me, could you provide more context? There are many notable people named Claude throughout history - for example:

- Claude Shannon (mathematician and "father of information theory")
- Claude Monet (French Impressionist painter)
- Claude Lévi-Strauss (French anthropologist)
- Various historical figures named Claude

Which Claude were you interested in learning about? Or were you asking about me as an AI assistant?